# Article eligibility and user profiles

Run notebooks 3_01 and 3_02 first.
**Input:** supplied histories, article vectors and chronological impressions/candidates.
**Output:** observation-time eligibility, static supplied-history profiles and causal profile checks.
Static profiles never include logged feedback. In replay, score every request at timestamp t before consuming its labels.
`long_text` is the supplied-history baseline; `text` uses the capped sequence, and category interests use all causal clicks.

Run the cells in order. Code must be available in this runtime, including `src/newsrec`.
For Colab, upload the full checkout or mount its folder on Drive; opening only the notebook does not upload the package.
Set `NEWSREC_CODE` before the setup cell if the checkout is elsewhere. Set `DATA_ROOT` for separate datasets.
Inputs are prepared parquet files under `1_Data_Preparation/processed`; outputs go to `3_Representations/artifacts`.
The calculation cells below are passed to shared storage helpers.

In [ ]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Code: run from the Group 6 checkout, or set NEWSREC_CODE to that folder.
CODE_ROOT = Path(os.environ['NEWSREC_CODE']) if 'NEWSREC_CODE' in os.environ else next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/newsrec').is_dir()
)
sys.path.insert(0, str(CODE_ROOT / 'src'))

# Data: use the checkout by default. For Drive, mount it first and set DATA_ROOT.
DATA_ROOT = None  # Colab example: '/content/drive/MyDrive/BT4222'
ROOT = Path(DATA_ROOT or os.getenv('NEWSREC_ROOT') or CODE_ROOT).expanduser().resolve()
ARTIFACTS = ROOT / '3_Representations/artifacts'
print('Code:', CODE_ROOT)
print('Data and outputs:', ROOT)

from newsrec.common import Settings, saved_stage_ready

from newsrec.representations import build_availability
from newsrec.pipeline import build_supplied_profiles, profile_preview
from collections import deque
import polars as pl
from newsrec.common import DEFAULT_SETTINGS, SPLITS
from newsrec.temporal import TemporalState as _StateStorage, User, HOUR
from newsrec.common import normalise

In [ ]:
SETTINGS = Settings()  # Fixed seed and dimensions; use the same settings in all stages.
REBUILD = False  # Set True after editing a calculation cell. Shared code changes invalidate caches.

## 1. Observation-time eligibility

`available_from` is the earliest observable slate appearance or supplied-history discovery, not publication time.
Do not backdate new test users' histories to training start. A history-only article has no observed exposure age.
Eligibility uses `available_from <= t`; it is separate from counts built from events strictly before `t`.

### Earliest observable article discovery

The minimum observed history/slate timestamp is an eligibility threshold. An article is served only when `available_from <= request_time`; this is not an invented publication date.

In [ ]:
def compute_availability(processed):
    histories = []
    exposures = []
    for split in SPLITS:
        histories.append(
            pl.scan_parquet(processed / f"impressions_{split}.parquet").select(
                "user_code", "timestamp", "history"
            )
        )
        exposures.append(
            pl.scan_parquet(processed / f"candidates_{split}.parquet").select(
                "news_code", "timestamp"
            )
        )
    first_hist = (
        pl.concat(histories)
        .group_by("user_code", "history")
        .agg(pl.col("timestamp").min())
        .filter(pl.col("history").is_not_null() & (pl.col("history") != ""))
        .with_columns(pl.col("history").str.split(" ").alias("news_id"))
        .explode("news_id", empty_as_null=True)
        .with_columns(
            pl.col("news_id").str.strip_prefix("N").cast(pl.Int32).alias("news_code")
        )
        .group_by("news_code")
        .agg(pl.col("timestamp").min().alias("history_discovered_at"))
    )
    first_exp = (
        pl.concat(exposures)
        .group_by("news_code")
        .agg(pl.col("timestamp").min().alias("first_observed_exposure"))
    )
    frame = (
        first_hist.join(first_exp, on="news_code", how="full", coalesce=True)
        .with_columns(
            pl.min_horizontal("history_discovered_at", "first_observed_exposure").alias(
                "available_from"
            )
        )
        .sort("news_code")
        .collect(engine="streaming")
        .to_pandas()
    )
    frame["discovery_source"] = np.where(
        frame.history_discovered_at.isna(),
        "slate",
        np.where(
            frame.first_observed_exposure.isna()
            | (frame.history_discovered_at < frame.first_observed_exposure),
            "history",
            "slate_or_tied",
        ),
    )
    start = min(
        pl.scan_parquet(processed / f"impressions_{s}.parquet")
        .select(pl.col("timestamp").min())
        .collect()
        .item()
        for s in SPLITS
    )
    frame["left_censored"] = frame.first_observed_exposure.eq(start)
    return frame, start

In [ ]:
if REBUILD or not saved_stage_ready(ROOT, SETTINGS, "availability"):
    availability = build_availability(
        ROOT, SETTINGS, availability_builder=compute_availability
    )
else:
    print("Loaded saved article availability.")
    availability = pd.read_parquet(ARTIFACTS / "article_availability.parquet")
display(availability.head())
display(availability.discovery_source.value_counts())

## 2. Supplied-history profiles

Pool every valid article in each supplied history; timestamps are unknown, so apply no decay.
Validate all distinct user/history pairs before pooling a user once.
`history_over_cap` means the supplied history exceeds the replay sequence cap; the static pool still uses the full history.
Only `seen_in_train` users may fit user clusters. Validation/test-only profiles are transform inputs at their own request times.

### Supplied-history pooling

Ignore unavailable/zero vectors, mean-pool the valid history vectors and normalise. Supplied histories have no click timestamps, so this static profile adds no invented recency decay.

In [ ]:
def normalised_mean(matrix, rows):
    if not len(rows):
        return (np.zeros(matrix.shape[1], dtype=np.float32), 0)
    selected = matrix[np.asarray(rows, dtype=np.int32)]
    valid = np.linalg.norm(selected, axis=1) > 1e-12
    if valid.any():
        pooled = normalise(selected[valid].mean(axis=0))
    else:
        pooled = np.zeros(matrix.shape[1], np.float32)
    return (pooled, int(valid.sum()))


def pool_supplied_history(history, bundle):
    raw = "" if pd.isna(history) else str(history)
    codes = [int(x.removeprefix("N")) for x in raw.split()]
    rows = bundle.rows(codes)
    text, valid_text = normalised_mean(bundle.text, rows) # text embeddings from 3_01 (text64.npy)
    entity, valid_entity = normalised_mean(bundle.entity, rows) # entity embeddings from 3_02 (article_entity100.npy)
    return rows, text, entity, valid_text, valid_entity

In [ ]:
if REBUILD or not saved_stage_ready(ROOT, SETTINGS, "profiles"):
    profiles = build_supplied_profiles(
        ROOT, SETTINGS, history_pooler=pool_supplied_history
    )
else:
    print("Loaded saved supplied-history profiles.")
    profiles = pd.read_parquet(ARTIFACTS / "supplied_profile_index.parquet")
display(
    profiles.groupby("seen_in_train").agg(
        users=("user_code", "size"),
        no_history=("history_length", lambda x: int(x.eq(0).sum())),
        no_entity_profile=("valid_entity_history_count", lambda x: int(x.eq(0).sum())),
    )
)
display(profiles.head())

## 3. Demonstrate causal profile updates

Seed each user once; append genuine earlier click events, retaining repeats. Score all requests in a timestamp
batch before consuming its feedback. Full category counts use all prior clicks; vector/NRMS histories are capped.
Snapshot chronology is unverified by default, so its capped subset is deterministic and not called recent.
Recent profiles use timestamped in-window clicks only. Up to three category interests are ranked by full
prior category counts and pooled from valid capped title vectors. Empty groups are masked.

The example processes only three initial timestamp batches, with their preceding state intact.

### Causal updates and full-history interests

`begin`: seed each user once and initialise per-category title-vector sums.
`profile`: return capped text/entity history, a recent profile and up to three category centroids.
Category sums include all earlier clicks even after their article rows leave the capped sequence.
`finish`: apply feedback only after the whole timestamp batch has been scored.
Snapshot click times/order are unknown; ID sorting provides deterministic retention, not inferred recency.
Memory grows with users and observed user/category pairs, rather than all clicked events.

In [ ]:
class TemporalState(_StateStorage):
    """Causal calculations; the base allocates state and the disk-backed pair store."""

    def begin(self, requests, t):
        t = pd.Timestamp(t)
        if self.active_time is not None:
            raise ValueError("Previous timestamp has not been finished")
        if self.last_update is not None and t <= self.last_update:
            raise ValueError(
                "Replay timestamps must increase strictly; merge same-time requests"
            )
        if t < self.start or t > self.end:
            raise ValueError("Request outside state horizon")
        if (
            requests.impression_uid.duplicated().any()
            or not requests.timestamp.eq(t).all()
        ):
            raise ValueError("begin requires unique requests from this timestamp only")
        self.active_time = t
        for row in requests.itertuples(index=False):
            text = "" if pd.isna(row.history) else str(row.history)
            codes = tuple(int(token.removeprefix("N")) for token in text.split())
            key = int(row.user_code)
            if key in self.users:
                if self.users[key].snapshot_key != codes:
                    raise ValueError(f"Supplied history changed for user {key}")
                continue
            snapshot = self.bundle.rows(codes)
            user = User(snapshot, codes, deque(maxlen=self.cfg.history_cap))
            user.categories.update(self.cat[snapshot])
            user.subcategories.update(self.subcat[snapshot])
            for article in snapshot:
                category = self.cat[article]
                if category not in user.interest_sums:
                    user.interest_sums[category] = np.zeros(self.cfg.text_dim, np.float32)
                user.interest_sums[category] += self.bundle.title[article]
            self.users[key] = user

    def profile(self, user_code, t):
        t = pd.Timestamp(t)
        if self.active_time != t:
            raise ValueError(
                "Call begin() for this timestamp before requesting profiles"
            )
        key = int(user_code)
        user = self.users[key]
        cache_key = (key, user.version, t.value)
        if cache_key in self.profiles:
            return self.profiles[cache_key]
        observed = list(user.observed)
        seed = (
            user.snapshot
            if self.cfg.snapshot_order_verified
            else np.sort(user.snapshot)
        )
        history = np.r_[seed, [a for _, a in observed]].astype(np.int32)[
            -self.cfg.history_cap :
        ]
        recent = [
            a for time, a in observed if t.value - time <= self.cfg.recent_hours * HOUR
        ][-self.cfg.recent_count :]
        text, valid_text = normalised_mean(self.bundle.text, history)
        entity, valid_entity = normalised_mean(self.bundle.entity, history)
        long, _ = normalised_mean(self.bundle.text, user.snapshot)
        recent_vector, valid_recent = normalised_mean(self.bundle.text, recent)
        interests = []
        interest_categories = []
        for cat, _ in sorted(user.categories.items(), key=lambda x: (-x[1], x[0]))[
            : self.cfg.interest_count
        ]:
            # Use all causal clicks, even when their rows leave the sequence cap.
            vector = user.interest_sums[cat]
            norm = np.linalg.norm(vector)
            if norm > 0:
                interests.append(vector / norm)
                interest_categories.append(cat)
        padded = np.zeros(self.cfg.history_cap, np.int32)
        padded[: len(history)] = history
        result = {
            "text": text,
            "entity": entity,
            "long_text": long,
            "recent_text": recent_vector,
            "interests": np.asarray(interests, np.float32).reshape(
                -1, self.cfg.text_dim
            ),
            "history_rows": padded,
            "history_mask": padded != 0,
            "selected_interest_categories": interest_categories,
            "recent_rows": list(map(int, recent)),
            "history_length": len(user.snapshot) + user.n_observed,
            "retained_length": len(history),
            "valid_text_fraction": valid_text / max(1, len(history)),
            "valid_entity_fraction": valid_entity / max(1, len(history)),
            "has_text": bool(np.linalg.norm(text) > 0),
            "has_entity": bool(np.linalg.norm(entity) > 0),
            "has_recent": bool(valid_recent and np.linalg.norm(recent_vector) > 0),
            "no_supplied_history": len(user.snapshot) == 0,
            "group_count": len(interests),
            "selected_group_count": min(self.cfg.interest_count, len(user.categories)),
            "state_version": user.version,
        }
        self.profiles[cache_key] = result
        return result

    def finish(self, requests, events, t, feedback_splits):
        t = pd.Timestamp(t)
        if self.active_time != t:
            raise ValueError("Timestamp is not active")
        if not events.timestamp.eq(t).all() or not set(events.impression_uid) <= set(
            requests.impression_uid
        ):
            raise ValueError("Feedback must belong to this timestamp's requests")
        owners = requests.set_index("impression_uid").user_code
        if not np.array_equal(
            events.user_code.to_numpy(), events.impression_uid.map(owners).to_numpy()
        ):
            raise ValueError("Feedback/request user mismatch")
        if events.duplicated(["impression_uid", "news_code"]).any():
            raise ValueError("Duplicate feedback event")
        for u in requests.user_code.unique():
            self.users[int(u)].last_impression = t
        selected = events[events.split.isin(feedback_splits)]
        if len(selected):
            if not selected.clicked.isin([0, 1]).all():
                raise ValueError("Feedback labels must be binary")
            rows = self.bundle.rows(selected.news_code)
            labels = selected.clicked.to_numpy(np.int64)
            self.pairs.update(selected.user_code, rows, labels, t)
            np.add.at(self.exposures, rows, 1)
            np.add.at(self.clicks, rows, labels)
            unseen = self.first[rows] < 0
            self.first[rows[unseen]] = t.value
            hour = int(t.value // HOUR - self.start_hour)
            np.add.at(self.hour_exposures[:, hour], rows, 1)
            np.add.at(self.hour_clicks[:, hour], rows, labels)
            self.total_exposures += len(rows)
            self.total_clicks += int(labels.sum())
            clicked = selected.loc[selected.clicked.eq(1)].sort_values(
                ["user_code", "news_code", "impression_uid"]
            )
            for event in clicked.itertuples(index=False):
                user = self.users[int(event.user_code)]
                a = self.bundle.lookup[int(event.news_code)]
                user.observed.append((t.value, a))
                user.n_observed += 1
                user.categories[self.cat[a]] += 1
                user.subcategories[self.subcat[a]] += 1
                category = self.cat[a]
                if category not in user.interest_sums:
                    user.interest_sums[category] = np.zeros(self.cfg.text_dim, np.float32)
                user.interest_sums[category] += self.bundle.title[a]
                user.version += 1
        self.profiles.clear()
        self.last_update = t
        self.active_time = None

In [ ]:
from newsrec.audits import check_temporal_profiles

# Output: deterministic regression checks on a tiny, explicit synthetic history.
# Verify dominant interests survive the cap and labels affect only the next timestamp.
display(check_temporal_profiles(TemporalState, SETTINGS))

# Output: an example from real prepared data, not a complete leakage proof.
display(profile_preview(ROOT, SETTINGS, state_factory=TemporalState))
display(pd.read_csv(ROOT / 'results/representation_coverage.csv'))
display(pd.read_csv(ROOT / 'results/representation_cost.csv').tail())

## Handoff

The shared `TemporalState` class exposes causal text/entity/recent profiles, category counts, interest vectors,
and padded history-row sequences/masks. The feature build writes per-request history references so
retrieval, attention rankers and future ablations can reconstruct the same profile. State is timestamp-scoped;
a final user vector must never be reused for an earlier request. The disk pair store is replay scratch space,
not a resumable checkpoint. Persisted resume support is not implemented in this version.